# Inspect Delta Tables

Use this notebook to inspect local Delta Lake outputs created by the streaming jobs. Start JupyterLab from the repository root and select the `Crypto Market Intelligence (.venv)` kernel.

In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / "streaming").exists() and (repo_root.parent / "streaming").exists():
    repo_root = repo_root.parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repository root: {repo_root}")

In [ ]:
from streaming.spark_session import create_spark_session

spark = create_spark_session("inspect-local-delta-tables")
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")

In [ ]:
DELTA_TABLES = {
    "bronze_trades": "./storage/bronze/bronze_market_trades_raw",
    "bronze_klines": "./storage/bronze/bronze_market_klines_raw",
    "bronze_tickers": "./storage/bronze/bronze_market_tickers_raw",
    "bronze_invalid_events": "./storage/bronze/bronze_market_invalid_events",
    "silver_trades": "./storage/silver/silver_market_trades",
    "silver_klines": "./storage/silver/silver_market_klines",
    "silver_tickers": "./storage/silver/silver_market_tickers",
}

for name, path in DELTA_TABLES.items():
    print(f"{name}: {path}")

In [ ]:
import pandas as pd
from pyspark.sql import functions as F


def delta_exists(path: str) -> bool:
    return (Path(path) / "_delta_log").exists()


def table_status() -> pd.DataFrame:
    rows = []
    for name, path in DELTA_TABLES.items():
        delta_log_exists = delta_exists(path)
        rows.append(
            {
                "table": name,
                "path": path,
                "delta_log_exists": delta_log_exists,
                "path_exists": Path(path).exists(),
            }
        )
    return pd.DataFrame(rows)


def inspect_delta_table(table_name: str, limit: int = 20, order_by: str | None = None):
    path = DELTA_TABLES[table_name]
    if not delta_exists(path):
        print(f"No Delta table found at {path}")
        print("Start the producer and the matching Bronze stream, then rerun this cell.")
        return None

    df = spark.read.format("delta").load(path)
    print(f"Path: {path}")
    print("Schema:")
    df.printSchema()
    print(f"Rows: {df.count()}")

    sample_df = df
    if order_by and order_by in df.columns:
        sample_df = sample_df.orderBy(F.col(order_by).desc())

    display(sample_df.limit(limit).toPandas())
    return df

In [ ]:
table_status()

In [ ]:
table_name = "bronze_trades"
df = inspect_delta_table(table_name, limit=20, order_by="kafka_timestamp")

In [ ]:
if df is not None:
    df.createOrReplaceTempView("selected_delta_table")
    spark.sql(
        """
        select
            topic,
            key as symbol,
            count(*) as row_count,
            min(kafka_timestamp) as first_kafka_timestamp,
            max(kafka_timestamp) as latest_kafka_timestamp
        from selected_delta_table
        group by topic, key
        order by row_count desc
        """
    ).toPandas()
else:
    print("No selected table loaded yet.")

In [ ]:
path = DELTA_TABLES[table_name]
if delta_exists(path):
    abs_path = str(Path(path).resolve())
    display(spark.sql(f"DESCRIBE DETAIL delta.`{abs_path}`").toPandas())
else:
    print(f"No Delta metadata found for {path}")

Stop the Spark session when you are done with the notebook.

In [ ]:
# spark.stop()